# Data Factory - Synthetic Dataset Generation

## 1. Setup & Configuration

In [36]:
import json
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

# Setup project paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from src.services.llm_services import load_config

load_dotenv()
config = load_config("src/config/config.yaml")

In [37]:
# Configure Tesseract path from .env
import pytesseract
tess_path = os.getenv("TESSERACT_PATH")
if tess_path:
    # 1. Set path for pytesseract library
    pytesseract.pytesseract.tesseract_cmd = tess_path
    
    # 2. Add Tesseract directory to system PATH for unstructured's subprocess calls
    tess_dir = os.path.dirname(tess_path)
    if tess_dir not in os.environ['PATH']:
        os.environ['PATH'] = tess_dir + os.pathsep + os.environ['PATH']
    print(f"Tesseract path injected: {tess_dir}")


## 2. Data Ingestion


In [38]:
file_path = config.get('source_pdf_path', './data/report.pdf')

if not Path(file_path).is_file():
    raise FileNotFoundError(f"❌ PDF not found at: {file_path}")

# Initialize and load
print("Extracting text via PyMuPDF...")
loader = PyMuPDFLoader(str(file_path))

# PyMuPDF loads the document as a list of LangChain Document objects (one per page)
pages = loader.load()
print(f"✅ Successfully loaded {len(pages)} pages.")

Extracting text via PyMuPDF...
✅ Successfully loaded 142 pages.


## 3. Text Chunking

In [42]:
# 1. Initialize standard splitter (using your config)
c_size = config.get('data_factory_chunk_size', 1500)
c_overlap = config.get('data_factory_chunk_overlap', 200)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = c_size,
    chunk_overlap = c_overlap,
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
)

print(f"Starting chunking with {c_size}-char size and {c_overlap}-char overlap...")

# 2. Split the documents directly
# 'docs' is the list of page documents returned by loader.load()
chunks = text_splitter.split_documents(pages)

print(f"✅ Total chunks created: {len(chunks)}")

# Optional: Print a sample to verify metadata survived
if chunks:
    print("\n--- Sample Chunk ---")
    print(f"Page Number: {chunks[5].metadata.get('page')}") 
    print(f"Text Preview: {chunks[5].page_content[:150]}...")

Starting chunking with 1500-char size and 200-char overlap...
✅ Total chunks created: 542

--- Sample Chunk ---
Page Number: 3
Text Preview: §240.10D-1(b). 
☐ 
 
Indicate by check mark whether the registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act). Yes ☐ No ☒ 
The ...


In [45]:
from json import JSONDecodeError
from langchain_core.documents import Document

chunks_json_path = config.get('chunks_json_path', 'artifacts/data_factory/chunks.json')

# Ensure directory exists
os.makedirs(os.path.dirname(chunks_json_path), exist_ok=True)

def _chunk_to_dict(chunk):
    return {
        'page_content': getattr(chunk, 'page_content', str(chunk)),
        'metadata': getattr(chunk, 'metadata', {}),
    }

if os.path.exists(chunks_json_path):
    try:
        with open(chunks_json_path, 'r', encoding='utf-8') as f:
            chunks_data = json.load(f)
        chunks = [
            Document(page_content=item['page_content'], metadata=item.get('metadata', {}))
            if isinstance(item, dict) and 'page_content' in item
            else item
            for item in chunks_data
        ]
        print(f"✅ Chunks loaded from {chunks_json_path}")
    except JSONDecodeError:
        print(f"⚠️ Invalid JSON cache at {chunks_json_path}; regenerating it.")
        serializable_chunks = [_chunk_to_dict(chunk) for chunk in chunks]
        with open(chunks_json_path, 'w', encoding='utf-8') as f:
            json.dump(serializable_chunks, f, indent=4, default=str)
        print(f"✅ Chunks saved to {chunks_json_path}")
else:
    serializable_chunks = [_chunk_to_dict(chunk) for chunk in chunks]
    with open(chunks_json_path, 'w', encoding='utf-8') as f:
        json.dump(serializable_chunks, f, indent=4, default=str)
    print(f"✅ Chunks saved to {chunks_json_path}")

⚠️ Invalid JSON cache at ./artifacts/data_factory/chunks.json; regenerating it.
✅ Chunks saved to ./artifacts/data_factory/chunks.json


## 4. Q&A Generation Setup

In [ ]:
# Initialize only OpenAI and the output parser
llm_openai = ChatOpenAI(model=config.get('llm_model', 'gpt-4o-mini'))

output_parser = StrOutputParser()

In [54]:
question_gen_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Lead AI Architect. Your task is to generate 10 high-quality questions based ONLY on the provided context.

    RULES:
    1. Every question MUST be answerable using ONLY the provided text.
    2. Do NOT use outside knowledge (like general financial stats for Uber).
    3. If the context is narrative, ask about goals, tone, or mission.
    4. If the context has data, ask about specific numbers and percentages.
    5. No intro/outro. Just 10 numbered questions."""),
    ("user", "Context: {context}")
])

answer_gen_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Senior Financial Analyst. 
    Your task is to answer the provided questions using ONLY the given context.

    You must return the output as a list of Question-Answer pairs.
    Format each pair exactly like this:
    Q: [The original question]
    A: [The concise answer based on context]

    Rules:
    1. Do not use outside knowledge.
    2. No citations or tags.
    3. If the answer is not in the context, write "Information not available".
    4. Provide exactly 10 pairs."""),
    ("user", "Context: {context}\n\nQuestions: {questions}")
])

# Use OpenAI for both chains
chain_openAI = question_gen_prompt | llm_openai | output_parser
chain_answer = answer_gen_prompt | llm_openai | output_parser

## 5. Dataset Generation

In [ ]:
output_file = config.get('dataset_json_path', './artifacts/data_factory/uber_dataset.json')
output_path = Path(output_file)
output_path.parent.mkdir(parents=True, exist_ok=True)

# Load existing dataset if present (checkpoint support)
if output_path.exists():
    with open(output_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
else:
    dataset = []

start_chunk = len(dataset)
batch_size = 50
total_chunks = len(chunks)

print(f"Starting generation from chunk {start_chunk} (total chunks: {total_chunks})")

processed_in_session = 0

while start_chunk < total_chunks:
    end_chunk = min(start_chunk + batch_size, total_chunks)
    print(f"Processing chunks {start_chunk} to {end_chunk}...")

    for current_idx in range(start_chunk, end_chunk):
        chunk = chunks[current_idx]
        try:
            metadata = chunk.metadata if hasattr(chunk, 'metadata') else chunk.get('metadata', {})
            page_content = chunk.page_content if hasattr(chunk, 'page_content') else chunk.get('page_content', '')

            if metadata.get('category') == 'Table' and metadata.get('text_as_html'):
                context_text = f"TABLE DATA (HTML):\n{metadata['text_as_html']}"
            else:
                context_text = page_content

            questions_text = chain_openAI.invoke({"context": context_text})
            qa_output = chain_answer.invoke({"context": context_text, "questions": questions_text})

            dataset.append({
                "chunk_id": current_idx,
                "context": context_text,
                "qa_pairs": qa_output,
            })

            processed_in_session += 1

            # Checkpoint every 10 processed items
            if processed_in_session % 10 == 0:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(dataset, f, indent=4, ensure_ascii=False)
                print(f"Saved checkpoint at processed {processed_in_session} (chunk {current_idx + 1})")

        except Exception as e:
            print(f"Error on chunk {current_idx}: {e}")
            # continue with next chunk
            continue

    # End of batch checkpoint
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=4, ensure_ascii=False)
    print(f"Batch complete. Saved checkpoint up to chunk {end_chunk}")

    # Move to next batch
    start_chunk = end_chunk

print("Generation complete for all chunks.")

Processing chunks 0 to 50...
Saved checkpoint at chunk 10
Saved checkpoint at chunk 20
Saved checkpoint at chunk 30
Saved checkpoint at chunk 40
Saved checkpoint at chunk 50
Generation complete for this batch.


## 6. Post-Processing & Splitting

In [56]:
output_file = config.get('dataset_json_path', './artifacts/data_factory/uber_dataset.json')
import json
import re
import random

with open(output_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

expanded_dataset = []

for item in raw_data:
    context = item["context"]
    qa_text = item["qa_pairs"]
    pairs = re.split(r'\n?\s?Q:', qa_text)
    
    for pair in pairs:
        if not pair.strip(): continue
        if 'A:' in pair:
            parts = pair.split('A:', 1)
            question = parts[0].strip().replace('Q:', '')
            answer = parts[1].strip()
            
            expanded_dataset.append({
                "context": context,
                "question": question,
                "answer":answer
            })

random.shuffle(expanded_dataset)
print(f"Total unique Q&A pairs: {len(expanded_dataset)}")

Total unique Q&A pairs: 500


In [ ]:
split_point = int(len(expanded_dataset) * 0.8)
train_set = expanded_dataset[:split_point]
test_set = expanded_dataset[split_point:]

def save_jsonl(data, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')
    print(f"Saved {len(data)} records to {filename}")
    
train_path = config.get('train_jsonl_path', './artifacts/train/train.jsonl')
test_path = config.get('golden_test_set_path', './artifacts/test/golden_test_set.jsonl')

save_jsonl(train_set, train_path)
save_jsonl(test_set, test_path)